# NBA API Endpoint Exploration

**Purpose**: Validate endpoints for each data category before building production scrapers.

**Approach**: One simple example per endpoint. If it works, we document it. If it breaks, we find alternatives.

## Setup

In [1]:
# Install if needed
# !pip install nba_api pandas

import time

import pandas as pd

# Set display options
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


# Rate limiting helper
def rate_limit(seconds=0.6):
    """NBA API rate limits aggressively. Always sleep between calls."""
    time.sleep(seconds)


print("Setup complete")

Setup complete


In [2]:
# Test IDs we'll use throughout
# Using well-known, stable IDs to minimize weird edge cases

TEST_PLAYER_ID = 201566  # Russell Westbrook (long career, lots of data)
TEST_TEAM_ID = 1610612744  # Golden State Warriors
TEST_GAME_ID = "0022300001"  # First game of 2023-24 season
TEST_SEASON = "2023-24"

print(f"Test Player ID: {TEST_PLAYER_ID}")
print(f"Test Team ID: {TEST_TEAM_ID}")
print(f"Test Game ID: {TEST_GAME_ID}")
print(f"Test Season: {TEST_SEASON}")

Test Player ID: 201566
Test Team ID: 1610612744
Test Game ID: 0022300001
Test Season: 2023-24


---
# 1. PLAY-BY-PLAY DATA

| Data Point | Purpose | Status |
|------------|---------|--------|
| Time-stamped scoring events | Competitive minutes filter, clutch stats | TBD |
| Score differential at each moment | Garbage time identification | TBD |
| Players on floor per possession | Lineup context, rotation patterns | TBD |
| Substitution timestamps | Minutes distribution patterns | TBD |
| Foul events with time | Foul trouble modeling | TBD |

## 1.2 PlayByPlayV3 - Newer version with player tracking

**Note**: V3 may have additional data but can be less stable. Let's compare.

In [42]:
from nba_api.stats.endpoints import boxscoresummaryv3

box_summary = boxscoresummaryv3.BoxScoreSummaryV3(game_id=TEST_GAME_ID)
rate_limit()

dfs = box_summary.get_data_frames()
print(f"Number of dataframes: {len(dfs)}")

for i, df in enumerate(dfs):
    print(f"\nDataFrame {i}: {df.shape}")
    if len(df) > 0:
        print(f"  Columns: {list(df.columns)}")

Number of dataframes: 9

DataFrame 0: (1, 13)
  Columns: ['gameId', 'gameCode', 'gameStatus', 'gameStatusText', 'period', 'gameClock', 'gameTimeUTC', 'gameEt', 'awayTeamId', 'homeTeamId', 'duration', 'attendance', 'sellout']

DataFrame 1: (1, 4)
  Columns: ['gameId', 'gameDate', 'attendance', 'gameDuration']

DataFrame 2: (1, 7)
  Columns: ['gameId', 'arenaId', 'arenaName', 'arenaCity', 'arenaState', 'arenaCountry', 'arenaTimezone']

DataFrame 3: (3, 7)
  Columns: ['gameId', 'personId', 'name', 'nameI', 'firstName', 'familyName', 'jerseyNum']

DataFrame 4: (2, 13)
  Columns: ['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'teamWins', 'teamLosses', 'period1Score', 'period2Score', 'period3Score', 'period4Score', 'score']

DataFrame 5: (7, 6)
  Columns: ['gameId', 'teamId', 'personId', 'firstName', 'familyName', 'jerseyNum']

DataFrame 6: (5, 20)
  Columns: ['recencyOrder', 'gameId', 'gameTimeUTC', 'gameEt', 'gameStatus', 'gameStatusText', 'awayTeamId', 'awayTeamCi

In [16]:
from nba_api.stats.endpoints import playbyplayv3

# Fetch play-by-play for a single game
pbp = playbyplayv3.PlayByPlayV3(game_id=TEST_GAME_ID)
rate_limit()

pbp_df = pbp.get_data_frames()[0]
print(f"Shape: {pbp_df.shape}")
print(f"\nColumns: {list(pbp_df.columns)}")
print("\n--- First 5 rows ---")
pbp_df.head()

Shape: (504, 24)

Columns: ['gameId', 'actionNumber', 'clock', 'period', 'teamId', 'teamTricode', 'personId', 'playerName', 'playerNameI', 'xLegacy', 'yLegacy', 'shotDistance', 'shotResult', 'isFieldGoal', 'scoreHome', 'scoreAway', 'pointsTotal', 'location', 'description', 'actionType', 'subType', 'videoAvailable', 'shotValue', 'actionId']

--- First 5 rows ---


,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,0022300001,2,PT12M00.00S,1,0,,0,,,0,0,0,,0,0,0,0,,Start of 1st Period (7:11 PM EST),period,start,0,0,1
1,0022300001,4,PT12M00.00S,1,1610612754,IND,1626167,Turner,M. Turner,0,0,0,,0,,,0,h,Jump Ball Turner vs. Allen: Tip to Toppin,Jump Ball,,1,0,2
2,0022300001,7,PT11M41.00S,1,1610612754,IND,1626167,Turner,M. Turner,2,21,2,Made,1,2,0,2,h,Turner 2' Cutting Dunk Shot (2 PTS) (Haliburto...,Made Shot,Cutting Dunk Shot,1,2,3
3,0022300001,9,PT11M23.00S,1,1610612739,CLE,1630596,Mobley,E. Mobley,59,53,8,Missed,1,,,0,v,MISS Mobley 8' Turnaround Jump Shot,Missed Shot,Turnaround Jump Shot,1,2,4
4,0022300001,10,PT11M20.00S,1,1610612754,IND,1626167,Turner,M. Turner,0,0,0,,0,,,0,h,Turner REBOUND (Off:0 Def:1),Rebound,Unknown,1,0,5


In [ ]:
# =============================================================================
# PlayByPlayV3 - FINAL SCHEMA REFERENCE
# =============================================================================

# Primary action type classification
ACTION_TYPES_V3 = {
    "Made Shot": "field_goal_made",
    "Missed Shot": "field_goal_missed",
    "Free Throw": "free_throw",
    "Rebound": "rebound",
    "Turnover": "turnover",
    "Foul": "foul",
    "Substitution": "substitution",
    "Timeout": "timeout",
    "Jump Ball": "jump_ball",
    "Violation": "violation",
    "period": "period_marker",
    "": "defensive_play",  # Steals and blocks - REQUIRES description parsing
}

# Subtype mappings by action type
SUBTYPES_V3 = {
    "field_goal_made": {
        # Shot types - useful for shot quality modeling
        "Jump Shot",
        "Pullup Jump shot",
        "Step Back Jump shot",
        "Running Jump Shot",
        "Turnaround Jump Shot",
        "Fadeaway Jump Shot",
        "Driving Layup Shot",
        "Driving Finger Roll Layup Shot",
        "Running Finger Roll Layup Shot",
        "Driving Reverse Layup Shot",
        "Driving Floating Jump Shot",
        "Cutting Dunk Shot",
        # ... more exist, capture dynamically
    },
    "free_throw": {
        "Free Throw 1 of 1",
        "Free Throw 1 of 2",
        "Free Throw 2 of 2",
        "Free Throw 1 of 3",
        "Free Throw 2 of 3",
        "Free Throw 3 of 3",
        # Technical, flagrant, clear path FTs exist too
    },
    "rebound": {
        "Unknown",  # 95% of rebounds - must infer ORB/DRB from context
        "Normal Rebound",
    },
    "turnover": {
        "Lost Ball",
        "Bad Pass",
        "Traveling",
        "Offensive Foul Turnover",
        "Out of Bounds",
        # ... more exist
    },
    "foul": {
        "Shooting",
        "Personal",
        "Offensive",
        "Loose Ball",
        "Technical",
        "Flagrant",
        "Away From Play",
    },
    "period_marker": {
        "start",
        "end",
    },
}

# Defensive plays (empty actionType) - parse from description
DEFENSIVE_PLAY_PATTERNS = {
    "steal": ["STEAL"],
    "block": ["BLOCK"],
}


def parse_defensive_play(description: str) -> str:
    """Parse steals/blocks from description field when actionType is empty."""
    if pd.isna(description):
        return "unknown"
    desc_upper = description.upper()
    for play_type, patterns in DEFENSIVE_PLAY_PATTERNS.items():
        if any(p in desc_upper for p in patterns):
            return play_type
    return "unknown"


# Key columns for scraping
PBP_V3_COLUMNS = {
    "identifiers": ["gameId", "actionNumber", "period"],
    "timing": ["clock", "timeActual"],  # clock is "PT10M29.00S" format
    "score": ["scoreHome", "scoreAway"],
    "action": ["actionType", "subType", "description"],
    "players": ["personId", "playerName"],  # Only primary player
    "team": ["teamId", "teamTricode"],
    "shot_detail": ["shotDistance", "xLegacy", "yLegacy", "shotResult", "isFieldGoal"],
}

# Computed fields you'll need to derive
DERIVED_FIELDS = {
    "score_margin": "scoreHome - scoreAway (from home team perspective)",
    "seconds_remaining": "Parse clock string -> total seconds in period",
    "game_seconds_elapsed": "(period-1)*720 + (720 - seconds_remaining)",  # Reg periods
    "is_clutch": "period >= 4 AND seconds_remaining <= 300 AND abs(score_margin) <= 5",
    "is_garbage_time": "Define threshold, e.g., |margin| > 20 in 4th",
    "rebound_type": "Compare rebounder teamId to previous missed shot teamId",
    "lineup_on_floor": "Track substitutions to reconstruct 5-man units",
}

print("PlayByPlayV3 Schema Reference loaded.")
print(f"  - {len(ACTION_TYPES_V3)} action types")
print(f"  - {len(PBP_V3_COLUMNS)} column groups")
print(f"  - {len(DERIVED_FIELDS)} fields to derive")

## 1.3 BoxScoreTraditionalV3 - Players on floor / starters

**Note**: Play-by-play doesn't directly tell you who's on the floor at each moment. You need to track substitutions or use lineup-specific endpoints.

In [22]:
from nba_api.stats.endpoints import boxscoretraditionalv3

box = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=TEST_GAME_ID)
rate_limit()

# This returns multiple dataframes
dfs = box.get_data_frames()
print(f"Number of dataframes returned: {len(dfs)}")
for i, df in enumerate(dfs):
    print(f"\nDataFrame {i}: {df.shape[0]} rows, {df.shape[1]} cols")
    if len(df) > 0:
        print(f"  Columns: {list(df.columns)[:10]}...")

Number of dataframes returned: 3

DataFrame 0: 28 rows, 34 cols
  Columns: ['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'personId', 'firstName', 'familyName', 'nameI']...

DataFrame 1: 4 rows, 26 cols
  Columns: ['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'minutes', 'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage']...

DataFrame 2: 2 rows, 26 cols
  Columns: ['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'minutes', 'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage']...


In [23]:
# =============================================================================
# BoxScoreTraditionalV3 - FINAL SCHEMA REFERENCE
# =============================================================================

# DataFrame 0: PlayerStats
PLAYER_STATS_V3 = {
    "identifiers": {
        "gameId": "Game identifier",
        "teamId": "Team identifier",
        "personId": "Player identifier",
    },
    "player_meta": {
        "firstName": "Player first name",
        "familyName": "Player last name",
        "nameI": 'Abbreviated name (e.g., "L. James")',
        "position": "Position IF starter, empty string if bench",  # THIS IS YOUR STARTER FLAG
        "jerseyNum": "Jersey number",
        "comment": "DNP reason / injury status",  # IMPORTANT: Check for "DNP", "DND", etc.
    },
    "counting_stats": {
        "minutes": 'Minutes played (string format, e.g., "PT34M12.00S")',  # NEEDS PARSING
        "points": "Points scored",
        "reboundsOffensive": "Offensive rebounds",
        "reboundsDefensive": "Defensive rebounds",
        "reboundsTotal": "Total rebounds",
        "assists": "Assists",
        "steals": "Steals",
        "blocks": "Blocks",
        "turnovers": "Turnovers",
        "foulsPersonal": "Personal fouls",
    },
    "shooting": {
        "fieldGoalsMade": "FGM",
        "fieldGoalsAttempted": "FGA",
        "fieldGoalsPercentage": "FG%",
        "threePointersMade": "3PM",
        "threePointersAttempted": "3PA",
        "threePointersPercentage": "3P%",
        "freeThrowsMade": "FTM",
        "freeThrowsAttempted": "FTA",
        "freeThrowsPercentage": "FT%",
    },
    "impact": {
        "plusMinusPoints": "Plus/minus for the game",  # KEY FEATURE
    },
}

# DataFrame 1: TeamStarterBenchStats
TEAM_STARTER_BENCH_V3 = {
    "split_field": "startersBench",  # Values: 'Starters' or 'Bench'
    "use_case": "Compare starter vs bench production, bench depth analysis",
}


# Starter detection logic
def is_starter(position: str) -> bool:
    """
    In V3, starters have position filled ('G', 'F', 'C', 'G-F', etc.)
    Bench players have empty string.
    """
    return position != "" and pd.notna(position)


# Minutes parsing (V3 uses ISO 8601 duration format)
def parse_minutes_v3(minutes_str: str) -> float:
    """
    Parse 'PT34M12.00S' -> 34.2 minutes
    Returns 0.0 for DNP/None
    """
    if pd.isna(minutes_str) or minutes_str == "" or minutes_str is None:
        return 0.0
    try:
        # Format: PT{minutes}M{seconds}S
        import re

        match = re.match(r"PT(\d+)M([\d.]+)S", minutes_str)
        if match:
            mins = int(match.group(1))
            secs = float(match.group(2))
            return mins + secs / 60
        return 0.0
    except:
        return 0.0


# DNP detection
def is_dnp(comment: str) -> bool:
    """Check if player did not play."""
    if pd.isna(comment) or comment == "":
        return False
    dnp_keywords = ["DNP", "DND", "NOT WITH TEAM", "INACTIVE"]
    return any(kw in comment.upper() for kw in dnp_keywords)


# =============================================================================
# WHAT YOU CAN COMPUTE FROM THIS
# =============================================================================
COMPUTABLE_METRICS = {
    "true_shooting_pct": "PTS / (2 * (FGA + 0.44 * FTA))",
    "efg_pct": "(FGM + 0.5 * 3PM) / FGA",
    "turnover_rate": "TOV / (FGA + 0.44 * FTA + TOV)",  # Simplified
    "free_throw_rate": "FTA / FGA",
    "three_point_rate": "3PA / FGA",
}

# =============================================================================
# WHAT YOU STILL NEED FROM BoxScoreAdvancedV3
# =============================================================================
NEED_ADVANCED_ENDPOINT = [
    "offensiveRating",  # Individual offensive rating
    "defensiveRating",  # Individual defensive rating
    "usagePercentage",  # Usage rate
    "assistPercentage",  # AST%
    "reboundPercentage",  # REB%
    "effectiveFieldGoalPercentage",  # eFG% (also computable)
    "trueShootingPercentage",  # TS% (also computable)
]

print("BoxScoreTraditionalV3 Schema Reference loaded.")
print(f"  - PlayerStats: {sum(len(v) for v in PLAYER_STATS_V3.values())} fields")
print(f"  - Computable from this: {len(COMPUTABLE_METRICS)} metrics")
print(f"  - Still need AdvancedV3 for: {len(NEED_ADVANCED_ENDPOINT)} metrics")

BoxScoreTraditionalV3 Schema Reference loaded.
  - PlayerStats: 29 fields
  - Computable from this: 5 metrics
  - Still need AdvancedV3 for: 7 metrics


### PLAY-BY-PLAY SUMMARY

| Data Point | Endpoint | Status | Notes |
|------------|----------|--------|-------|
| Time-stamped scoring events | `PlayByPlayV2` | ✅ | EVENTMSGTYPE 1,3 + PCTIMESTRING |
| Score differential | `PlayByPlayV2` | ✅ | SCOREHOME, SCOREAWAY columns |
| Players on floor | **DERIVED** | ⚠️ | Must track subs from PBP or use lineup endpoints |
| Substitution timestamps | `PlayByPlayV2` | ✅ | EVENTMSGTYPE 8 |
| Foul events | `PlayByPlayV2` | ✅ | EVENTMSGTYPE 6 |

---
# 2. ADVANCED PLAYER METRICS

| Data Point | Purpose | Source |
|------------|---------|--------|
| Usage rate | Shot volume prediction | nba_api or compute |
| True shooting % | Efficiency signal | Compute from box score |
| Offensive rating (individual) | Scoring efficiency | nba_api |
| Defensive rating (individual) | Matchup context | nba_api |
| Assist rate | Playmaking role | nba_api or compute |
| Turnover rate | Ball-handling role | Compute |
| Rebound rate (ORB%, DRB%) | Rebounding role | Compute |

## 2.1 LeagueDashPlayerStats - Basic advanced metrics

In [ ]:
from nba_api.stats.endpoints import leaguedashplayerstats

# Per-game stats with advanced metrics
player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
    season=TEST_SEASON,
    per_mode_detailed="PerGame",
    measure_type_detailed_defense="Base",  # Can also be 'Advanced'
)
rate_limit()

player_stats_df = player_stats.get_data_frames()[0]
print(f"Shape: {player_stats_df.shape}")
print(f"\nColumns: {list(player_stats_df.columns)}")

In [ ]:
# Now try with measure_type='Advanced' - this gets us the good stuff
player_adv = leaguedashplayerstats.LeagueDashPlayerStats(
    season=TEST_SEASON, per_mode_detailed="PerGame", measure_type_detailed_defense="Advanced"
)
rate_limit()

player_adv_df = player_adv.get_data_frames()[0]
print(f"Advanced Stats Shape: {player_adv_df.shape}")
print(f"\nColumns: {list(player_adv_df.columns)}")

In [ ]:
# Check for our target metrics
target_cols = [
    "PLAYER_NAME",
    "USG_PCT",
    "TS_PCT",
    "OFF_RATING",
    "DEF_RATING",
    "AST_PCT",
    "TM_TOV_PCT",
    "OREB_PCT",
    "DREB_PCT",
]
available = [c for c in target_cols if c in player_adv_df.columns]
missing = [c for c in target_cols if c not in player_adv_df.columns]

print(f"Available target columns: {available}")
print(f"Missing target columns: {missing}")

if available:
    print(f"\n--- Sample data for {TEST_PLAYER_ID} ---")
    sample = player_adv_df[player_adv_df["PLAYER_ID"] == TEST_PLAYER_ID][available]
    display(sample)

## 2.2 PlayerDashboardByYearOverYear - Individual player deep dive

In [25]:
from nba_api.stats.endpoints import playerdashboardbyyearoveryear

player_dash = playerdashboardbyyearoveryear.PlayerDashboardByYearOverYear(
    player_id=TEST_PLAYER_ID, per_mode_detailed="PerGame"
)
rate_limit()

dfs = player_dash.get_data_frames()
print(f"Number of dataframes: {len(dfs)}")
for i, df in enumerate(dfs):
    print(f"\nDF {i}: {df.shape}")
    if len(df) > 0:
        print(f"  Sample cols: {list(df.columns)[:8]}...")

Number of dataframes: 2

DF 0: (1, 65)
  Sample cols: ['GROUP_SET', 'GROUP_VALUE', 'TEAM_ID', 'TEAM_ABBREVIATION', 'MAX_GAME_DATE', 'GP', 'W', 'L']...

DF 1: (20, 65)
  Sample cols: ['GROUP_SET', 'GROUP_VALUE', 'TEAM_ID', 'TEAM_ABBREVIATION', 'MAX_GAME_DATE', 'GP', 'W', 'L']...


## 2.3 BoxScoreAdvancedV3 - Per-game advanced stats

In [24]:
from nba_api.stats.endpoints import boxscoreadvancedv3

box_adv = boxscoreadvancedv3.BoxScoreAdvancedV3(game_id=TEST_GAME_ID)
rate_limit()

dfs = box_adv.get_data_frames()
print(f"Number of dataframes: {len(dfs)}")

# DataFrame 0: Player stats
box_adv_df = dfs[0]
print(f"\nPlayerStats Shape: {box_adv_df.shape}")
print(f"\nColumns: {list(box_adv_df.columns)}")

# V3 uses camelCase, not UPPER_SNAKE_CASE
# Adjust column names accordingly
v3_display_cols = [
    "firstName",
    "familyName",
    "teamTricode",
    "minutes",
    "offensiveRating",
    "defensiveRating",
    "usagePercentage",
    "trueShootingPercentage",
    "assistPercentage",
]

# Filter to columns that exist
available_cols = [c for c in v3_display_cols if c in box_adv_df.columns]
missing_cols = [c for c in v3_display_cols if c not in box_adv_df.columns]

if missing_cols:
    print(f"\n⚠️  Missing expected columns: {missing_cols}")

print("\n--- First 5 players ---")
display(box_adv_df[available_cols].head())

# Check DataFrame 1 if it exists (usually team-level stats)
if len(dfs) > 1:
    print("\n--- DataFrame 1 (Team Stats) ---")
    print(f"Shape: {dfs[1].shape}")
    print(f"Columns: {list(dfs[1].columns)}")

Number of dataframes: 2

PlayerStats Shape: (28, 37)

Columns: ['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'personId', 'firstName', 'familyName', 'nameI', 'playerSlug', 'position', 'comment', 'jerseyNum', 'minutes', 'estimatedOffensiveRating', 'offensiveRating', 'estimatedDefensiveRating', 'defensiveRating', 'estimatedNetRating', 'netRating', 'assistPercentage', 'assistToTurnover', 'assistRatio', 'offensiveReboundPercentage', 'defensiveReboundPercentage', 'reboundPercentage', 'turnoverRatio', 'effectiveFieldGoalPercentage', 'trueShootingPercentage', 'usagePercentage', 'estimatedUsagePercentage', 'estimatedPace', 'pace', 'pacePer40', 'possessions', 'PIE']

--- First 5 players ---


,firstName,familyName,teamTricode,minutes,offensiveRating,defensiveRating,usagePercentage,trueShootingPercentage,assistPercentage
0,Max,Strus,CLE,28:17,119.7,116.7,0.164,0.506,0.000
1,Evan,Mobley,CLE,35:51,124.7,110.4,0.181,0.538,0.167
2,Jarrett,Allen,CLE,21:07,104.3,106.4,0.154,0.683,0.000
3,Donovan,Mitchell,CLE,36:39,119.5,117.9,0.337,0.748,0.391
4,Darius,Garland,CLE,31:59,94.4,114.7,0.213,0.549,0.316



--- DataFrame 1 (Team Stats) ---
Shape: (2, 30)
Columns: ['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'minutes', 'estimatedOffensiveRating', 'offensiveRating', 'estimatedDefensiveRating', 'defensiveRating', 'estimatedNetRating', 'netRating', 'assistPercentage', 'assistToTurnover', 'assistRatio', 'offensiveReboundPercentage', 'defensiveReboundPercentage', 'reboundPercentage', 'estimatedTeamTurnoverPercentage', 'turnoverRatio', 'effectiveFieldGoalPercentage', 'trueShootingPercentage', 'usagePercentage', 'estimatedUsagePercentage', 'estimatedPace', 'pace', 'pacePer40', 'possessions', 'PIE']


In [ ]:
# =============================================================================
# BoxScoreAdvancedV3 - FINAL SCHEMA REFERENCE
# =============================================================================

# DataFrame 0: PlayerStats (Advanced)
PLAYER_ADVANCED_V3 = {
    "identifiers": {
        "gameId": "Game identifier",
        "teamId": "Team identifier",
        "personId": "Player identifier",
    },
    "player_meta": {
        "firstName": "Player first name",
        "familyName": "Player last name",
        "nameI": "Abbreviated name",
        "position": "Position IF starter, empty if bench",
        "comment": "DNP reason / injury status",
        "jerseyNum": "Jersey number",
    },
    "ratings": {
        "offensiveRating": "Points produced per 100 possessions",
        "defensiveRating": "Points allowed per 100 possessions",
        "netRating": "ORTG - DRTG",
        "estimatedOffensiveRating": "NBA estimated ORTG",  # Slightly different calc
        "estimatedDefensiveRating": "NBA estimated DRTG",
        "estimatedNetRating": "NBA estimated net rating",
    },
    "percentages": {
        "usagePercentage": "USG% - % of team plays used while on floor",
        "trueShootingPercentage": "TS% - scoring efficiency",
        "effectiveFieldGoalPercentage": "eFG% - FG% adjusted for 3PT value",
        "assistPercentage": "AST% - % of teammate FGs assisted",
        "offensiveReboundPercentage": "ORB% - % of available offensive rebounds grabbed",
        "defensiveReboundPercentage": "DRB% - % of available defensive rebounds grabbed",
        "reboundPercentage": "TRB% - total rebound percentage",
    },
    "other": {
        "assistToTurnover": "AST/TO ratio",
        "assistRatio": "AST ratio (per 100 possessions)",
        "turnoverRatio": "TOV ratio (turnovers per 100 possessions)",
        "pace": "Possessions per 48 minutes",
        "possessions": "Total possessions played",
        "PIE": "Player Impact Estimate (0-1 scale, ~.100 is average)",
    },
    "estimated_variants": {
        "estimatedUsagePercentage": "NBA estimated USG%",
        "estimatedPace": "NBA estimated pace",
    },
}

# DataFrame 1: TeamStats (Advanced) - same columns minus player-specific ones
TEAM_ADVANCED_V3 = {
    "rows": "2 (one per team)",
    "use_case": "Game-level team efficiency, pace for both teams",
    "key_fields": ["offensiveRating", "defensiveRating", "pace", "possessions"],
}


# =============================================================================
# IMPORTANT: Minutes format differs from TraditionalV3!
# =============================================================================
def parse_minutes_advanced_v3(minutes_str: str) -> float:
    """
    AdvancedV3 uses 'MM:SS' format (e.g., '28:17')
    TraditionalV3 uses 'PT28M17.00S' format

    Handle both for safety.
    """
    if pd.isna(minutes_str) or minutes_str == "" or minutes_str is None:
        return 0.0

    minutes_str = str(minutes_str)

    # Try MM:SS format first (AdvancedV3)
    if ":" in minutes_str and "PT" not in minutes_str:
        try:
            parts = minutes_str.split(":")
            mins = int(parts[0])
            secs = int(parts[1]) if len(parts) > 1 else 0
            return mins + secs / 60
        except:
            pass

    # Try ISO 8601 format (TraditionalV3)
    if "PT" in minutes_str:
        try:
            import re

            match = re.match(r"PT(\d+)M([\d.]+)S", minutes_str)
            if match:
                mins = int(match.group(1))
                secs = float(match.group(2))
                return mins + secs / 60
        except:
            pass

    return 0.0


# =============================================================================
# ESTIMATED vs ACTUAL - WHICH TO USE?
# =============================================================================
ESTIMATED_VS_ACTUAL = """
NBA provides both 'estimated' and actual versions of some metrics.

Differences:
- Actual: Calculated directly from box score events
- Estimated: Uses statistical models to account for lineup context

Recommendation:
- For TRAINING: Use actual (offensiveRating, defensiveRating, usagePercentage)
- The estimated versions add noise and are less interpretable

Exception:
- If you're doing lineup-adjusted analysis, estimated may be more appropriate
- But for standard player evaluation, stick with actual
"""

# =============================================================================
# COLUMNS TO SCRAPE (FINAL LIST)
# =============================================================================
ADVANCED_V3_SCRAPE_COLUMNS = [
    # Identifiers
    "gameId",
    "teamId",
    "personId",
    # Time
    "minutes",
    # Core ratings (USE THESE)
    "offensiveRating",
    "defensiveRating",
    "netRating",
    # Core percentages (USE THESE)
    "usagePercentage",
    "trueShootingPercentage",
    "effectiveFieldGoalPercentage",
    "assistPercentage",
    "offensiveReboundPercentage",
    "defensiveReboundPercentage",
    # Other useful
    "turnoverRatio",
    "pace",
    "possessions",
    "PIE",
]

# Skip these (redundant or less useful)
ADVANCED_V3_SKIP_COLUMNS = [
    "estimatedOffensiveRating",  # Use actual
    "estimatedDefensiveRating",  # Use actual
    "estimatedNetRating",  # Use actual
    "estimatedUsagePercentage",  # Use actual
    "estimatedPace",  # Use actual
    "pacePer40",  # Redundant with pace
    "assistToTurnover",  # Can compute from Traditional
    "assistRatio",  # assistPercentage is more standard
    "reboundPercentage",  # Have ORB% and DRB% separately
]

print("BoxScoreAdvancedV3 Schema Reference loaded.")
print(f"  - PlayerStats: {sum(len(v) for v in PLAYER_ADVANCED_V3.values() if isinstance(v, dict))} fields")
print(f"  - Recommended scrape columns: {len(ADVANCED_V3_SCRAPE_COLUMNS)}")
print(f"  - Skip columns: {len(ADVANCED_V3_SKIP_COLUMNS)}")

### ADVANCED PLAYER METRICS SUMMARY

| Data Point | Endpoint | Status | Notes |
|------------|----------|--------|-------|
| Usage rate | `LeagueDashPlayerStats(Advanced)` | ✅ | USG_PCT |
| True shooting % | `LeagueDashPlayerStats(Advanced)` | ✅ | TS_PCT |
| Offensive rating | `BoxScoreAdvancedV2` | ✅ | OFF_RATING (per-game granularity) |
| Defensive rating | `BoxScoreAdvancedV2` | ✅ | DEF_RATING (per-game granularity) |
| Assist rate | `LeagueDashPlayerStats(Advanced)` | ✅ | AST_PCT |
| Turnover rate | `LeagueDashPlayerStats(Advanced)` | ⚠️ | TM_TOV_PCT (team), individual needs compute |
| Rebound rates | `LeagueDashPlayerStats(Advanced)` | ✅ | OREB_PCT, DREB_PCT |

---
# 3. ADVANCED TEAM METRICS

| Data Point | Purpose | Source |
|------------|---------|--------|
| Pace | Counting stat inflation/deflation | nba_api or compute |
| Team offensive rating | Scoring environment | nba_api or compute |
| Team defensive rating | Opponent context | nba_api or compute |
| Opponent pace | Game tempo prediction | Derived |
| Opponent defensive rating | Matchup difficulty | Derived |

## 3.1 LeagueDashTeamStats - Team-level stats

In [26]:
from nba_api.stats.endpoints import leaguedashteamstats

team_stats = leaguedashteamstats.LeagueDashTeamStats(
    season=TEST_SEASON, per_mode_detailed="PerGame", measure_type_detailed_defense="Advanced"
)
rate_limit()

team_stats_df = team_stats.get_data_frames()[0]
print(f"Shape: {team_stats_df.shape}")
print(f"\nColumns: {list(team_stats_df.columns)}")

Shape: (30, 46)

Columns: ['TEAM_ID', 'TEAM_NAME', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'E_OFF_RATING', 'OFF_RATING', 'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT', 'EFG_PCT', 'TS_PCT', 'E_PACE', 'PACE', 'PACE_PER40', 'POSS', 'PIE', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'OFF_RATING_RANK', 'DEF_RATING_RANK', 'NET_RATING_RANK', 'AST_PCT_RANK', 'AST_TO_RANK', 'AST_RATIO_RANK', 'OREB_PCT_RANK', 'DREB_PCT_RANK', 'REB_PCT_RANK', 'TM_TOV_PCT_RANK', 'EFG_PCT_RANK', 'TS_PCT_RANK', 'PACE_RANK', 'PIE_RANK']


In [27]:
# Check for pace, off/def rating
target_cols = ["TEAM_NAME", "PACE", "OFF_RATING", "DEF_RATING", "NET_RATING"]
available = [c for c in target_cols if c in team_stats_df.columns]

print("Target team metrics:")
display(team_stats_df[available].head(10))

Target team metrics:


,TEAM_NAME,PACE,OFF_RATING,DEF_RATING,NET_RATING
0,Atlanta Hawks,100.84,116.4,118.4,-2.0
1,Boston Celtics,97.98,122.2,110.6,11.7
2,Brooklyn Nets,97.56,112.4,115.4,-2.9
3,Charlotte Hornets,97.81,108.6,119.2,-10.6
4,Chicago Bulls,96.94,114.0,115.7,-1.7
5,Cleveland Cavaliers,97.62,114.7,112.1,2.5
6,Dallas Mavericks,100.60,117.0,114.9,2.1
7,Denver Nuggets,97.43,117.8,112.3,5.5
8,Detroit Pistons,100.45,109.0,118.0,-9.0
9,Golden State Warriors,99.91,116.9,114.5,2.4


## 3.2 TeamDashboardByGeneralSplits - Detailed team breakdowns

In [28]:
from nba_api.stats.endpoints import teamdashboardbygeneralsplits

team_dash = teamdashboardbygeneralsplits.TeamDashboardByGeneralSplits(
    team_id=TEST_TEAM_ID,
    season=TEST_SEASON,
    per_mode_detailed="PerGame",
    measure_type_detailed_defense="Advanced",
)
rate_limit()

dfs = team_dash.get_data_frames()
print(f"Number of dataframes: {len(dfs)}")
for i, df in enumerate(dfs):
    print(f"\nDF {i}: {df.shape}")
    if len(df) > 0 and "GROUP_VALUE" in df.columns:
        print(f"  Groups: {list(df['GROUP_VALUE'].unique())}")

Number of dataframes: 6

DF 0: (1, 47)
  Groups: ['2023-24']

DF 1: (2, 47)
  Groups: ['Home', 'Road']

DF 2: (2, 47)
  Groups: ['Wins', 'Losses']

DF 3: (7, 47)
  Groups: ['October', 'November', 'December', 'January', 'February', 'March', 'April']

DF 4: (2, 47)
  Groups: ['Pre All-Star', 'Post All-Star']

DF 5: (5, 47)
  Groups: ['0 Days Rest', '1 Days Rest', '2 Days Rest', '3 Days Rest', '6+ Days Rest']


In [29]:
# Overall team stats (usually first dataframe)
overall = dfs[0]
print("=== TEAM OVERALL STATS ===")
print(f"Columns: {list(overall.columns)}")
if "OFF_RATING" in overall.columns:
    print(f"\nOff Rating: {overall['OFF_RATING'].values}")
    print(f"Def Rating: {overall['DEF_RATING'].values}")
    print(f"Pace: {overall['PACE'].values}")

=== TEAM OVERALL STATS ===
Columns: ['GROUP_SET', 'GROUP_VALUE', 'SEASON_YEAR', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'E_OFF_RATING', 'OFF_RATING', 'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING', 'NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'TM_TOV_PCT', 'EFG_PCT', 'TS_PCT', 'E_PACE', 'PACE', 'PACE_PER40', 'POSS', 'PIE', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'OFF_RATING_RANK', 'DEF_RATING_RANK', 'NET_RATING_RANK', 'AST_PCT_RANK', 'AST_TO_RANK', 'AST_RATIO_RANK', 'OREB_PCT_RANK', 'DREB_PCT_RANK', 'REB_PCT_RANK', 'TM_TOV_PCT_RANK', 'EFG_PCT_RANK', 'TS_PCT_RANK', 'PACE_RANK', 'PIE_RANK']

Off Rating: [116.9]
Def Rating: [114.5]
Pace: [99.91]


## 3.3 BoxScoreAdvancedV2 (Team level) - Per-game team metrics

In [35]:
# We already loaded this above, but let's look at team-level data (usually index 1)
box_adv = boxscoreadvancedv3.BoxScoreAdvancedV3(game_id=TEST_GAME_ID)
rate_limit()

dfs = box_adv.get_data_frames()
print(f"Number of dataframes: {len(dfs)}")

# Index 1 is usually team stats
if len(dfs) > 1:
    team_box = dfs[0]
    print(f"\nTeam Box Score Shape: {team_box.shape}")
    print(f"Columns: {list(team_box.columns)}")
    display(team_box)

Number of dataframes: 2

Team Box Score Shape: (28, 37)
Columns: ['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'personId', 'firstName', 'familyName', 'nameI', 'playerSlug', 'position', 'comment', 'jerseyNum', 'minutes', 'estimatedOffensiveRating', 'offensiveRating', 'estimatedDefensiveRating', 'defensiveRating', 'estimatedNetRating', 'netRating', 'assistPercentage', 'assistToTurnover', 'assistRatio', 'offensiveReboundPercentage', 'defensiveReboundPercentage', 'reboundPercentage', 'turnoverRatio', 'effectiveFieldGoalPercentage', 'trueShootingPercentage', 'usagePercentage', 'estimatedUsagePercentage', 'estimatedPace', 'pace', 'pacePer40', 'possessions', 'PIE']


,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,estimatedOffensiveRating,offensiveRating,estimatedDefensiveRating,defensiveRating,estimatedNetRating,netRating,assistPercentage,assistToTurnover,assistRatio,offensiveReboundPercentage,defensiveReboundPercentage,reboundPercentage,turnoverRatio,effectiveFieldGoalPercentage,trueShootingPercentage,usagePercentage,estimatedUsagePercentage,estimatedPace,pace,pacePer40,possessions,PIE
0,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1629622,Max,Strus,M. Strus,max-strus,F,,,28:17,119.7,119.7,116.7,116.7,3.0,3.0,0.000,0.00,0.0,0.000,0.036,0.018,9.1,0.500,0.506,0.164,0.164,102.68,102.68,85.56,61.0,0.026
1,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1630596,Evan,Mobley,E. Mobley,evan-mobley,F,,,35:51,124.7,124.7,110.4,110.4,14.3,14.3,0.167,2.50,25.0,0.071,0.222,0.156,10.0,0.538,0.538,0.181,0.181,103.10,103.10,85.91,77.0,0.133
2,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1628386,Jarrett,Allen,J. Allen,jarrett-allen,C,,,21:07,104.3,104.3,106.4,106.4,-2.1,-2.1,0.000,0.00,0.0,0.091,0.263,0.171,12.5,0.667,0.683,0.154,0.154,106.84,106.84,89.03,47.0,0.124
3,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1628378,Donovan,Mitchell,D. Mitchell,donovan-mitchell,G,,,36:39,119.5,119.5,117.9,117.9,1.5,1.5,0.391,3.00,24.3,0.031,0.108,0.072,8.1,0.714,0.748,0.337,0.337,101.50,101.50,84.58,77.0,0.224
4,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1629636,Darius,Garland,D. Garland,darius-garland,G,,,31:59,94.4,94.4,114.7,114.7,-20.3,-20.3,0.316,1.50,26.1,0.000,0.000,0.000,17.4,0.455,0.549,0.213,0.213,105.04,105.04,87.53,72.0,0.094
5,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1627777,Georges,Niang,G. Niang,georges-niang,,,,27:08,125.4,125.4,110.2,110.2,15.3,15.3,0.042,0.00,9.1,0.000,0.200,0.114,0.0,0.556,0.607,0.164,0.164,104.40,104.40,87.00,59.0,0.076
6,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1630171,Isaac,Okoro,I. Okoro,isaac-okoro,,,,23:01,96.0,96.0,131.4,131.4,-35.4,-35.4,0.125,2.00,33.3,0.000,0.056,0.026,16.7,0.833,0.833,0.077,0.077,105.32,105.32,87.77,50.0,0.037
7,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1627747,Caris,LeVert,C. LeVert,caris-levert,,,,30:24,111.6,111.6,114.7,114.7,-3.1,-3.1,0.167,4.00,21.1,0.000,0.200,0.098,5.3,0.417,0.472,0.205,0.205,108.15,108.15,90.12,69.0,0.075
8,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,1629731,Dean,Wade,D. Wade,dean-wade,,,,2:46,12.5,12.5,125.0,125.0,-112.5,-112.5,0.000,0.00,0.0,0.000,0.000,0.000,0.0,0.000,0.000,0.000,0.000,138.80,138.80,115.66,8.0,-0.100
9,0022300001,1610612739,Cleveland,Cavaliers,CLE,cavaliers,202684,Tristan,Thompson,T. Thompson,tristan-thompson,,,,2:48,28.6,28.6,166.7,166.7,-138.1,-138.1,0.000,0.00,0.0,0.000,0.500,0.143,0.0,0.000,0.000,0.000,0.000,111.43,111.43,92.86,7.0,0.000


### ADVANCED TEAM METRICS SUMMARY

| Data Point | Endpoint | Status | Notes |
|------------|----------|--------|-------|
| Pace | `LeagueDashTeamStats(Advanced)` | ✅ | PACE column |
| Team offensive rating | `LeagueDashTeamStats(Advanced)` | ✅ | OFF_RATING |
| Team defensive rating | `LeagueDashTeamStats(Advanced)` | ✅ | DEF_RATING |
| Opponent pace | **DERIVED** | ⚠️ | Join with opponent team stats |
| Opponent def rating | **DERIVED** | ⚠️ | Join with opponent team stats |

---
# 4. PLAYER CONTEXT DATA

| Data Point | Purpose | Source |
|------------|---------|--------|
| Position | Positional matchups, rotation depth | nba_api |
| Starter flag | Role identification | nba_api (box score) |
| Height / weight | Physical matchup context | nba_api |
| Age | Performance trajectory | nba_api |
| Years experience | Role stability | nba_api |
| Draft position | Talent proxy | nba_api |

## 4.1 CommonPlayerInfo - Player biographical data

In [ ]:
from nba_api.stats.endpoints import commonplayerinfo

player_info = commonplayerinfo.CommonPlayerInfo(player_id=TEST_PLAYER_ID)
rate_limit()

player_info_df = player_info.get_data_frames()[0]
print(f"Shape: {player_info_df.shape}")
print(f"\nColumns: {list(player_info_df.columns)}")

In [ ]:
# Extract our target fields
target_cols = [
    "DISPLAY_FIRST_LAST",
    "POSITION",
    "HEIGHT",
    "WEIGHT",
    "BIRTHDATE",
    "SEASON_EXP",
    "DRAFT_YEAR",
    "DRAFT_ROUND",
    "DRAFT_NUMBER",
]
available = [c for c in target_cols if c in player_info_df.columns]

print("=== PLAYER CONTEXT DATA ===")
display(player_info_df[available].T)

## 4.2 CommonAllPlayers - All players in a season (bulk)

In [ ]:
from nba_api.stats.endpoints import commonallplayers

all_players = commonallplayers.CommonAllPlayers(is_only_current_season=1, league_id="00", season=TEST_SEASON)
rate_limit()

all_players_df = all_players.get_data_frames()[0]
print(f"Shape: {all_players_df.shape}")
print(f"\nColumns: {list(all_players_df.columns)}")
print("\n--- First 5 players ---")
all_players_df.head()

## 4.3 DraftHistory - Draft position data

In [ ]:
from nba_api.stats.endpoints import drafthistory

draft = drafthistory.DraftHistory(season_year_nullable="2023")
rate_limit()

draft_df = draft.get_data_frames()[0]
print(f"Shape: {draft_df.shape}")
print(f"\nColumns: {list(draft_df.columns)}")
print("\n--- First 10 picks ---")
draft_df.head(10)

## 4.4 BoxScoreTraditionalV3 - Starter flag per game

In [41]:
# Already loaded above - let's check START_POSITION
from nba_api.stats.endpoints import boxscoretraditionalv3

box = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id="0020900005")
rate_limit()

player_box = box.get_data_frames()[0]

print("=== STARTER IDENTIFICATION (V3) ===")
print("'position' column values:")
print(player_box["position"].value_counts())

print("\nStarters (non-empty position):")
starters = player_box[player_box["position"] != ""][["firstName", "familyName", "teamTricode", "position"]]
display(starters)

print("\nBench players (empty position):")
bench = player_box[player_box["position"] == ""][["firstName", "familyName", "teamTricode", "minutes"]]
display(bench)

=== STARTER IDENTIFICATION (V3) ===
'position' column values:
position
G    10
F     8
      4
C     2
Name: count, dtype: int64

Starters (non-empty position):


,firstName,familyName,teamTricode,position
0,Marvin,Williams,ATL,F
1,Josh,Smith,ATL,F
2,Al,Horford,ATL,C
3,Joe,Johnson,ATL,G
4,Mike,Bibby,ATL,G
5,Jamal,Crawford,ATL,F
6,Zaza,Pachulia,ATL,G
7,Maurice,Evans,ATL,F
8,Jeff,Teague,ATL,F
9,Joe,Smith,ATL,G



Bench players (empty position):


,firstName,familyName,teamTricode,minutes
10,Jason,Collins,ATL,
11,Randolph,Morris,ATL,
22,Josh,McRoberts,IND,
23,AJ,Price,IND,


### PLAYER CONTEXT SUMMARY

| Data Point | Endpoint | Status | Notes |
|------------|----------|--------|-------|
| Position | `CommonPlayerInfo` | ✅ | POSITION column |
| Starter flag | `BoxScoreTraditionalV2` | ✅ | START_POSITION != '' |
| Height | `CommonPlayerInfo` | ✅ | HEIGHT (string format) |
| Weight | `CommonPlayerInfo` | ✅ | WEIGHT (string format) |
| Age | `CommonPlayerInfo` | ⚠️ | BIRTHDATE (compute age from this) |
| Years experience | `CommonPlayerInfo` | ✅ | SEASON_EXP |
| Draft position | `CommonPlayerInfo` or `DraftHistory` | ✅ | DRAFT_NUMBER, DRAFT_ROUND |

---
# 5. BONUS: Game Schedule / Scoreboard Data

You'll need this to know which games to scrape.

In [38]:
from nba_api.stats.endpoints import leaguegamefinder

# Find all games for a team in a season
games = leaguegamefinder.LeagueGameFinder(
    team_id_nullable=TEST_TEAM_ID,
    season_nullable=TEST_SEASON,
    season_type_nullable="Regular Season",
)
rate_limit()

games_df = games.get_data_frames()[0]
print(f"Games found: {len(games_df)}")
print(f"\nColumns: {list(games_df.columns)}")
print("\n--- First 5 games ---")
games_df[["GAME_ID", "GAME_DATE", "MATCHUP", "WL", "PTS", "PLUS_MINUS"]].head()

Games found: 82

Columns: ['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']

--- First 5 games ---


,GAME_ID,GAME_DATE,MATCHUP,WL,PTS,PLUS_MINUS
0,0022301198,2024-04-14,GSW vs. UTA,W,123,7.0
1,0022301182,2024-04-12,GSW vs. NOP,L,109,-5.0
2,0022301169,2024-04-11,GSW @ POR,W,100,8.0
3,0022301155,2024-04-09,GSW @ LAL,W,134,14.0
4,0022301142,2024-04-07,GSW vs. UTA,W,118,8.0


In [37]:
from nba_api.stats.endpoints import scoreboardv2

# Get scoreboard for a specific date
scoreboard = scoreboardv2.ScoreboardV2(game_date="2024-01-15")
rate_limit()

dfs = scoreboard.get_data_frames()
print(f"Number of dataframes: {len(dfs)}")

# Game header is usually the first one
if len(dfs) > 0:
    game_header = dfs[0]
    print(f"\nGame Header Shape: {game_header.shape}")
    print(f"Columns: {list(game_header.columns)}")
    display(game_header.head())

Number of dataframes: 9

Game Header Shape: (11, 18)
Columns: ['GAME_DATE_EST', 'GAME_SEQUENCE', 'GAME_ID', 'GAME_STATUS_ID', 'GAME_STATUS_TEXT', 'GAMECODE', 'HOME_TEAM_ID', 'VISITOR_TEAM_ID', 'SEASON', 'LIVE_PERIOD', 'LIVE_PC_TIME', 'NATL_TV_BROADCASTER_ABBREVIATION', 'HOME_TV_BROADCASTER_ABBREVIATION', 'AWAY_TV_BROADCASTER_ABBREVIATION', 'LIVE_PERIOD_TIME_BCAST', 'ARENA_NAME', 'WH_STATUS', 'WNBA_COMMISSIONER_FLAG']


,GAME_DATE_EST,GAME_SEQUENCE,GAME_ID,GAME_STATUS_ID,GAME_STATUS_TEXT,GAMECODE,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,LIVE_PERIOD,LIVE_PC_TIME,NATL_TV_BROADCASTER_ABBREVIATION,HOME_TV_BROADCASTER_ABBREVIATION,AWAY_TV_BROADCASTER_ABBREVIATION,LIVE_PERIOD_TIME_BCAST,ARENA_NAME,WH_STATUS,WNBA_COMMISSIONER_FLAG
0,2024-01-15T00:00:00,1,0022300555,3,Final,20240115/HOUPHI,1610612755,1610612745,2023,4,,NBA TV,NBCSP,SCHN,Q4 - NBA TV,Wells Fargo Center,1,0
1,2024-01-15T00:00:00,2,0022300556,3,Final,20240115/NOPDAL,1610612742,1610612740,2023,4,,None,BSSW-DAL,BSNO,Q4 -,American Airlines Center,1,0
2,2024-01-15T00:00:00,3,0022300557,3,Final,20240115/ORLNYK,1610612752,1610612753,2023,4,,None,MSG,BSFL,Q4 -,Madison Square Garden,1,0
3,2024-01-15T00:00:00,4,0022300558,3,Final,20240115/DETWAS,1610612764,1610612765,2023,4,,None,MNMT,BSDET,Q4 -,Capital One Arena,1,0
4,2024-01-15T00:00:00,5,0022300559,3,Final,20240115/SASATL,1610612737,1610612759,2023,4,,TNT,None,None,Q4 - TNT,State Farm Arena,1,0


---
# 6. MASTER SUMMARY

## Endpoints to Use

| Category | Primary Endpoint | Backup/Alternative |
|----------|------------------|--------------------|
| Play-by-Play | `PlayByPlayV2` | `PlayByPlayV3` (unstable) |
| Player Box Scores | `BoxScoreTraditionalV2` | - |
| Player Advanced (per-game) | `BoxScoreAdvancedV2` | - |
| Player Advanced (season) | `LeagueDashPlayerStats(Advanced)` | - |
| Team Advanced (season) | `LeagueDashTeamStats(Advanced)` | `TeamDashboardByGeneralSplits` |
| Player Bio | `CommonPlayerInfo` | `CommonAllPlayers` (bulk) |
| Draft Data | `DraftHistory` | `CommonPlayerInfo` |
| Game Finder | `LeagueGameFinder` | `ScoreboardV2` |

## Data That Requires Computation

1. **Players on floor at each moment** - Track substitutions from PBP
2. **Garbage time identification** - Compute from score differential + time remaining
3. **Individual turnover rate** - Compute: TOV / (FGA + 0.44*FTA + TOV)
4. **Opponent stats** - Join game data with opponent team's season stats
5. **Age** - Compute from BIRTHDATE

## Key Gotchas

1. **Rate limiting**: Always sleep 0.5-1.0s between calls
2. **Season format**: Use '2023-24' not '2024'
3. **Game ID format**: 10 digits, e.g., '0022300001'
4. **Height format**: Returns as string '6-3', needs parsing
5. **Multiple dataframes**: Many endpoints return lists of DFs
6. **Empty results**: Some seasons/players return empty - handle gracefully

In [ ]:
print("Endpoint exploration complete!")
print("\nNext steps:")
print("1. Build individual scrapers for each validated endpoint")
print("2. Add error handling and retry logic")
print("3. Implement rate limiting at scraper level")
print("4. Build computation functions for derived metrics")
print("5. Design database schema to store all this data")